# gridqueue w Google Colab

[![Otwórz w Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-matysik/gridqueue/blob/main/notebooks/gridqueue_colab.ipynb)

Ten notatnik uruchamia **pełny przebieg badawczy bez instalowania czegokolwiek lokalnie**: pobiera
dwie edycje ustawowego ujawnienia przyłączeniowego wprost od publikującego, przekształca je w wiersze
wspólnego schematu, ocenia jakość ujawnienia i odtwarza analizę wzdłużną opisaną w artykule.

**Co zobaczysz po uruchomieniu wszystkich komórek**

1. dwa dokumenty urzędowe pobrane z adresów publikującego, z sumami kontrolnymi;
2. wiersze w 21-polowym schemacie wyprowadzonym ze struktury obowiązku prawnego;
3. raport ośmiu reguł kontroli jakości;
4. przepływy w kolejce między edycjami wraz z **precyzją dopasowania** zmierzoną na polach
   wyłączonych z klucza;
5. porównanie odtworzonych liczb z wartościami opublikowanymi w artykule — komórka **przerywa
   wykonanie**, jeśli się rozjadą.

Czas pełnego przebiegu na darmowej maszynie Colab: poniżej minuty. Akcelerator nie jest potrzebny.

**Czego ten notatnik nie robi.** Nie odtwarza dokładności wydobycia wobec zbioru odniesienia, bo ten
wymaga odczytu przez człowieka, ani nie rozstrzyga współrzędnych, bo gazeter geometryczny jest
osobnym pobraniem na licencji ODbL. Jedno i drugie opisuje dokumentacja.

## 1. Instalacja

Jedno polecenie, z **oznaczonego wydania**, żeby wynik nie zależał od stanu gałęzi głównej.

In [ ]:
%pip install -q git+https://github.com/s-matysik/gridqueue@v1.2.1

import gridqueue as gq
print('gridqueue', gq.__version__, '| wersja schematu', gq.SCHEMA_VERSION)

## 2. Pobranie ujawnienia wprost od publikującego

Pobieramy dwie kolejne edycje wykazu operatora przesyłowego, wydawane na podstawie art. 7 ust. 8l
ustawy Prawo energetyczne. Obowiązek wyznacza **minimalną** częstość kwartalną, a ten publikujący
wydaje ponadto migawki miesięczne — z nich korzystamy.

Jeśli któryś adres zwróci 404, to znaczy, że publikujący wymienił plik: edycje przesuwają się
w czasie i starsze bywają usuwane. Komórka wypisze wtedy, co zrobić.

In [ ]:
import hashlib, pathlib, requests

BAZA = 'https://www.pse.pl/documents/20182/51490/'
WZOR = 'Informacje_publikowane_zgodnie_z_Art_7_ust_8l_ustawy_PE_stan_na_{}.xlsx'
EDYCJE = {'2026-07-31': '31_07_2026', '2026-08-31': '31_08_2026'}

NAGL = {'User-Agent': 'gridqueue-colab/1.2 (statutory grid connection disclosure)'}
RADA = ("  Pobierz bieżący wykaz ze strony https://www.pse.pl, wgraj go przez panel plików\n"
        "  Colab i wpisz jego nazwę do słownika `pliki` poniżej, na przykład:\n"
        "      pliki['2026-08-31'] = pathlib.Path('moj_wykaz.xlsx')")

pliki = {}
for etykieta, znacznik in EDYCJE.items():
    url = BAZA + WZOR.format(znacznik)
    try:
        r = requests.get(url, headers=NAGL, timeout=180)
    except requests.exceptions.SSLError:
        # Typowe w sieciach firmowych i instytucjonalnych z serwerem pośredniczącym,
        # który podmienia łańcuch certyfikatów. W Colab nie występuje.
        print(f'{etykieta}: weryfikacja certyfikatu nie powiodła się — najpewniej sieć\n'
              f'  z serwerem pośredniczącym. Nie obchodzimy tej kontroli.')
        print(RADA)
        continue
    except requests.exceptions.RequestException as e:
        print(f'{etykieta}: połączenie nieudane ({type(e).__name__}).')
        print(RADA)
        continue
    if r.status_code != 200 or len(r.content) < 10_000:
        # Edycje przesuwają się w czasie; starsze bywają usuwane po wymianie.
        print(f'{etykieta}: HTTP {r.status_code} — publikujący wymienił plik.')
        print(RADA)
        continue
    sciezka = pathlib.Path(f'pse_{znacznik}.xlsx')
    sciezka.write_bytes(r.content)
    pliki[etykieta] = sciezka
    print(f'{etykieta}: {len(r.content):,} B | sha256 {hashlib.sha256(r.content).hexdigest()[:16]}…')

if len(pliki) < 2:
    print('\nBrakuje edycji. Uzupełnij słownik `pliki` wgranymi dokumentami,')
    print('a następnie uruchom kolejne komórki — reszta notatnika ich nie pobiera.')
else:
    print('\nObie edycje pobrane.')

## 3. Rozpoznanie publikującego i wydobycie wierszy

Publikujący jest **rozpoznawany z zawartości dokumentu**, nie podawany ręcznie. Wydobycie opiera się
na położeniu słów i geometrii kolumn, a nie na dopasowaniu wzorców tekstowych.

In [ ]:
from gridqueue import detect_publisher, get_adapter

ramki = {}
for etykieta, sciezka in pliki.items():
    klucz = detect_publisher(str(sciezka))
    wynik = get_adapter(klucz).parse(str(sciezka))
    ramki[etykieta] = wynik.frame
    print(f'{etykieta}: rozpoznano {klucz} | {len(wynik.frame)} wierszy, {wynik.frame.shape[1]} kolumn')

ramki['2026-08-31'].head(3).T

## 4. Co zawiera schemat

Dwadzieścia jeden pól odwzorowuje **strukturę obowiązku prawnego**, nie układ dokumentu: każde pole
niesie odsyłacz do punktu przepisu, który je uzasadnia. Kolumny zaczynające się od podkreślenia to
rozszerzenia zadeklarowane przez publikującego, poza schematem wspólnym.

In [ ]:
from gridqueue import FIELDS, SCHEMA_VERSION, schema_table

df = ramki['2026-08-31']
pokrycie = (df[list(FIELDS)].notna().mean() * 100).round(2).sort_values(ascending=False)
print(f'schemat {SCHEMA_VERSION}: {len(FIELDS)} pól, wypełnionych {(pokrycie > 0).sum()}')
print(f'kolumn rozszerzeń: {sum(1 for c in df.columns if c.startswith("_"))}')
pokrycie[pokrycie > 0].to_frame('pokrycie [%]')

## 5. Reguły kontroli jakości

Osiem reguł wyprowadzonych z przepisu, nie z oczekiwań analityka. Każda zwraca **liczbę naruszeń,
nie wyjątek**, bo naruszenie jest właściwością ujawnienia, a nie błędem wykonania — narzędzie mierzy
jakość źródła, zamiast ją ukrywać.

In [ ]:
from gridqueue import run_quality

raport = run_quality(df)
print('naruszeń łącznie:', raport.as_dict()['naruszen_lacznie'], 'na', len(df), 'wierszy')
raport.as_frame()

## 6. Analiza wzdłużna — przepływy w kolejce

To jest część, dla której potrzebne były dwie edycje. Jedna rzecz stoi na drodze i jest ustaleniem
samym w sobie: **rejestr nie publikuje trwałego identyfikatora wniosku**, a zastępnik oparty na
liczbie porządkowej jest bezwartościowy między edycjami, bo wiersze się przesuwają. Dopasowanie idzie
więc po **kluczu treściowym**, z którego celowo wyłączono daty i status.

In [ ]:
from gridqueue import compare_editions

delta = compare_editions(ramki['2026-07-31'], ramki['2026-08-31'], '2026-07-31', '2026-08-31')
d = delta.as_dict()
for k in ('wierszy_a', 'wierszy_b', 'wspolnych', 'nowych', 'ubylych', 'zmian_statusu',
          'udzial_przejsc_w_przod', 'nowe_zamkniete', 'ubyle_czynne', 'migawka_monotoniczna'):
    print(f'{k}: {d[k]}')

delta.przejscia

### Dlaczego macierz przejść jest testem, a nie tylko wynikiem

Jeśli klucz treściowy dopasowuje właściwe wiersze, przejścia powinny biec zgodnie z kierunkiem
postępowania administracyjnego. Klucz dopasowujący losowo nie wytworzyłby tego porządku, więc wysoki
udział przejść w przód jest **niezależnym świadectwem poprawności dopasowania**.

Drugi wynik jest ostrzeżeniem dla każdego, kto chciałby czytać różnicę między edycjami jako zmianę
stanu kolejki: część wierszy pojawia się już w stanie zamkniętym, a część wierszy o statusie czynnym
znika. Rejestr **nie jest monotoniczną migawką** i narzędzie zgłasza to flagą.

## 7. Precyzja dopasowania

Świadectwo pośrednie domykamy bezpośrednim. Audyt porównuje każdą dopasowaną parę na **dziewięciu
polach wyłączonych z klucza** — nazwa podmiotu, nazwa obiektu, data złożenia wniosku i sześć pól
rozbicia mocy na technologie. Klucz nie wymusza zgodności żadnego z nich, więc ich zgodność jest
świadectwem niezależnym.

In [ ]:
from gridqueue import linkage_audit

audyt = linkage_audit(ramki['2026-07-31'], ramki['2026-08-31'])
a = audyt.as_dict()
print(f"par: {a['par']} | potwierdzonych {a['potwierdzonych']}, "
      f"niepewnych {a['niepewnych']}, fałszywych {a['falszywych']}")
print(f"precyzja: {a['precyzja_dolna']*100:.2f}–{a['precyzja_gorna']*100:.2f} %")
print('\nrozkład rozbieżności na polach wstrzymanych:')
audyt.rozbieznosci['kategoria'].value_counts().to_frame('liczba')

## 8. Zgodność z liczbami opublikowanymi w artykule

Ta komórka jest odpowiednikiem asercji z katalogu `reproducibility/`: **przerywa wykonanie**, jeśli
odtworzone liczby rozejdą się z opublikowanymi. Rozjazd nie musi oznaczać błędu — publikujący mógł
wymienić edycję — ale musi być widoczny, a nie przemilczany.

In [ ]:
OPUBLIKOWANE = {
    'wierszy_a': 892, 'wierszy_b': 876, 'wspolnych': 804, 'nowych': 45, 'ubylych': 57,
    'zmian_statusu': 58, 'nowe_zamkniete': 13, 'ubyle_czynne': 33, 'przejscia_wstecz': 1,
    'migawka_monotoniczna': False,
}
OPUBLIKOWANY_AUDYT = {'par': 804, 'potwierdzonych': 795, 'niepewnych': 6, 'falszywych': 3}

rozjazd = {k: (v, d[k]) for k, v in OPUBLIKOWANE.items() if d[k] != v}
rozjazd |= {k: (v, a[k]) for k, v in OPUBLIKOWANY_AUDYT.items() if a[k] != v}

if rozjazd:
    for k, (opub, otrzym) in rozjazd.items():
        print(f'ROZJAZD {k}: w artykule {opub}, odtworzono {otrzym}')
    raise AssertionError('odtworzone liczby nie odpowiadają opublikowanym — patrz wyżej')
print('Wszystkie liczby zgodne z artykułem.')
print(f"udział przejść w przód: {d['udzial_przejsc_w_przod']*100:.2f} %")
print(f"precyzja dopasowania: {a['precyzja_dolna']*100:.2f}–{a['precyzja_gorna']*100:.2f} %")

## 9. Analiza przekrojowa — dlaczego nie na tym publikującym

Przykład przekrojowy z artykułu mierzy czas od złożenia wniosku do wydania warunków. Kryterium
włączenia postawiono **przed** policzeniem wyniku: pokrycie obu dat powyżej 80 % i co najmniej
30 obserwacji w grupie. Ten publikujący kryterium **nie spełnia** — komórka niżej pokazuje dlaczego,
i jest to uczciwsze niż policzenie mediany na kilkunastu procentach rekordów.

Żeby uruchomić analizę przekrojową, wgraj dokument publikującego, który kryterium spełnia, przez
panel plików Colab i podaj jego ścieżkę w komórce.

In [ ]:
for etykieta, r in ramki.items():
    pokr = {p: round(float(r[p].notna().mean()) * 100, 2) for p in ('data_wniosku', 'data_warunkow')}
    kwalifikuje = all(v > 80 for v in pokr.values())
    stan = 'spełnione' if kwalifikuje else 'NIESPEŁNIONE'
    print(f"{etykieta}: data_wniosku {pokr['data_wniosku']} %, "
          f"data_warunkow {pokr['data_warunkow']} % -> kryterium 80 %: {stan}")

# Po wgraniu kwalifikującego dokumentu:
#   df2 = get_adapter(detect_publisher('moj_plik.pdf')).parse('moj_plik.pdf').frame
# a następnie uruchom examples/research_use_case.py na wyeksportowanym panelu.

## 10. Eksport

Panel zapisujemy w formacie kolumnowym i jako CSV. W Colab oba pliki są do pobrania z panelu plików.

In [ ]:
df.to_parquet('panel_pse_2026-08-31.parquet', index=False)
df.to_csv('panel_pse_2026-08-31.csv', index=False)
delta.przejscia.to_csv('przejscia_statusu.csv')
audyt.rozbieznosci.to_csv('audyt_dopasowania.csv', index=False)
print('zapisano: panel (parquet, csv), macierz przejść, rozbieżności audytu')

---

**Licencje.** Kod pakietu: Apache-2.0. Dokumenty źródłowe są publikacjami urzędowymi i nie są przez
pakiet redystrybuowane — notatnik pobiera je wprost od publikującego. Gazeter geometryczny, gdyby go
użyć, podlega ODbL z obowiązkiem atrybucji i dziedziczenia. Szczegóły w `DATA_LICENSE.md`.

**Dokumentacja:** https://s-matysik.github.io/gridqueue/ · **Repozytorium:** https://github.com/s-matysik/gridqueue